# Flat evaluation reporting demo

Demonstrates **Layer S** (structural row pairing), **Layer 1** (applicability), and **Layer 2** (value matching) on a small figure checklist:

- 3 figures, each with 2 panels
- Gold vs pred with realistic errors (reordered panels, wrong polarity, missing/spurious rows, label typo)

This notebook wires together the phase 1–4 modules directly (the phase 5 orchestrator is not implemented yet).

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from dataclasses import asdict
from pathlib import Path
from typing import Any

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from soda_mmqc.core.applicability_and_matching import report_layers
from soda_mmqc.core.eval_manifest import MatchingMetric, load_eval_manifest
from soda_mmqc.core.leaves import compare_strings, exact_primitive_similarity
from soda_mmqc.core.object_list_pairing import align_object_rows
from soda_mmqc.core.structural_reporting import build_by_list

In [ ]:
def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "soda_mmqc").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    return here


ROOT = find_repo_root()
DEMO_DIR = ROOT / "notebooks/fixtures/flat-eval-demo"

manifest = load_eval_manifest(DEMO_DIR / "manifest.json")
schema = json.loads((DEMO_DIR / "schema.json").read_text())
gold = json.loads((DEMO_DIR / "gold.json").read_text())
pred = json.loads((DEMO_DIR / "pred.json").read_text())

print(f"Checklist: {manifest.checklist}")
print(f"Profiled leaves: {manifest.profiled_leaf_properties()}")
print(f"Panel alignment keys: {manifest.alignment_keys_for('panels')}")

## Fixture overview

| Figure | Gold | Pred (errors) |
|--------|------|---------------|
| 1 | Perfect match | Perfect match |
| 2 | Panels 2A/2B | Reordered; `2A.is_micrograph` wrong; `figure_label` typo |
| 3 | Panels 3A/3B | 3A correct; 3B missing; spurious 3C |

In [ ]:
pd.set_option("display.max_colwidth", 60)

display(pd.DataFrame({"schema": [json.dumps(schema, indent=2)]}))
display(pd.DataFrame({"manifest": [json.dumps(json.loads((DEMO_DIR / "manifest.json").read_text()), indent=2)]}))
display(pd.DataFrame({"gold": [json.dumps(gold, indent=2)], "pred": [json.dumps(pred, indent=2)]}))

In [ ]:
PANEL_FIELDS = ("label", "is_micrograph", "caption_snippet")

LAYER_S_ORDER = ("correct_row", "missing_row", "spurious_row")
LAYER1_ORDER = (
    "correct_NA",
    "correct_applicable",
    "withheld_applicable",
    "spurious_applicable",
)
LAYER2_ORDER = ("TP", "TN", "FP", "FN", "match", "mismatch")

OUTCOME_COLORS = {
    "correct_row": "#2ca02c",
    "missing_row": "#ff7f0e",
    "spurious_row": "#d62728",
    "correct_NA": "#aec7e8",
    "correct_applicable": "#2ca02c",
    "withheld_applicable": "#ff7f0e",
    "spurious_applicable": "#d62728",
    "TP": "#2ca02c",
    "TN": "#98df8a",
    "FP": "#d62728",
    "FN": "#ff7f0e",
    "match": "#2ca02c",
    "mismatch": "#d62728",
}


def score_profiled_leaf(pred_value: Any, exp_value: Any, profile) -> float:
    """Leaf score for one profiled field (mirrors phase 5 leaf scoring)."""
    if profile.matching_metric == MatchingMetric.GRADED_STRING:
        if not isinstance(exp_value, str) or not isinstance(pred_value, str):
            return exact_primitive_similarity(pred_value, exp_value)
        return compare_strings(
            pred_value,
            exp_value,
            mode=profile.string_compare,
        ).score
    return exact_primitive_similarity(pred_value, exp_value)


def run_demo_evaluation(gold_doc: dict, pred_doc: dict, manifest):
    """Mini orchestrator: per-figure panel pairing + gold-indexed leaf reporting."""
    layer_s_rows: list[dict] = []
    layer_s_counts: Counter = Counter()
    instance_rows: list[dict] = []

    for fig_idx, (gold_fig, pred_fig) in enumerate(
        zip(gold_doc["figures"], pred_doc["figures"])
    ):
        gold_panels = gold_fig["panels"]
        pred_panels = pred_fig["panels"]

        pairing = align_object_rows(
            gold_panels,
            pred_panels,
            list_name="panels",
            manifest=manifest,
        )
        by_list = build_by_list(
            pairing,
            n_gold=len(gold_panels),
            n_pred=len(pred_panels),
        )

        for outcome, count in asdict(by_list.row_counts).items():
            layer_s_counts[outcome] += count

        for row in by_list.rows:
            layer_s_rows.append(
                {
                    "figure": fig_idx,
                    "gold_index": row.gold_index,
                    "pred_index": row.pred_index,
                    "structural": row.structural.value,
                    "similarity": row.similarity,
                }
            )

        def record_instance(instance_path: str, exp_value, pred_value, profile):
            score = score_profiled_leaf(pred_value, exp_value, profile)
            reporting = report_layers(exp_value, pred_value, profile, score)
            instance_rows.append(
                {
                    "instance_path": instance_path,
                    "gold": exp_value,
                    "pred": pred_value,
                    "score": score,
                    "layer1": reporting.layer1.value,
                    "layer2": reporting.layer2.value if reporting.layer2 else None,
                }
            )

        figure_label_profile = manifest.profile_for("figures[].figure_label")
        record_instance(
            f"figures[{fig_idx}].figure_label",
            gold_fig["figure_label"],
            pred_fig["figure_label"],
            figure_label_profile,
        )

        for gold_idx in range(len(gold_panels)):
            pred_idx = pairing.pred_index_for_gold(gold_idx)
            for field in PANEL_FIELDS:
                profile = manifest.profile_for(f"figures[].panels[].{field}")
                exp_value = gold_panels[gold_idx].get(field)
                pred_value = (
                    pred_panels[pred_idx].get(field) if pred_idx is not None else None
                )
                record_instance(
                    f"figures[{fig_idx}].panels[{gold_idx}].{field}",
                    exp_value,
                    pred_value,
                    profile,
                )

    instances_df = pd.DataFrame(instance_rows)
    layer_s_df = pd.DataFrame(layer_s_rows)
    layer1_counts = Counter(instances_df["layer1"])
    layer2_counts = Counter(
        v for v in instances_df["layer2"].dropna() if v is not None
    )

    return {
        "layer_s_counts": layer_s_counts,
        "layer_s_df": layer_s_df,
        "instances_df": instances_df,
        "layer1_counts": layer1_counts,
        "layer2_counts": layer2_counts,
    }


def counts_to_frame(counts: Counter, order: tuple[str, ...], label: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            label: [outcome for outcome in order],
            "count": [counts.get(outcome, 0) for outcome in order],
        }
    )


def plot_outcome_bar(
    counts: Counter,
    order: tuple[str, ...],
    *,
    title: str,
    x_label: str = "outcome",
) -> go.Figure:
    labels = list(order)
    values = [counts.get(label, 0) for label in labels]
    colors = [OUTCOME_COLORS.get(label, "#7f7f7f") for label in labels]
    fig = go.Figure(
        go.Bar(
            x=labels,
            y=values,
            marker_color=colors,
            text=values,
            textposition="outside",
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title=x_label,
        yaxis_title="count",
        showlegend=False,
        yaxis=dict(rangemode="tozero"),
    )
    return fig

In [ ]:
results = run_demo_evaluation(gold, pred, manifest)
instances_df = results["instances_df"]
layer_s_df = results["layer_s_df"]

print("Aggregated instance counts:")
display(instances_df.groupby(["layer1", "layer2"], dropna=False).size().reset_index(name="count"))

## Layer S — structural row reporting (`by_list`)

Row-slot outcomes for `panels[]` alignment (Hungarian pairing on `label`).
Spurious pred rows do not receive gold-indexed leaf instances.

In [ ]:
layer_s_summary = counts_to_frame(
    results["layer_s_counts"], LAYER_S_ORDER, "structural"
)
display(layer_s_summary)
display(layer_s_df.sort_values(["figure", "gold_index", "pred_index"], na_position="last"))

fig_s = plot_outcome_bar(
    results["layer_s_counts"],
    LAYER_S_ORDER,
    title="Layer S — panel row outcomes (all figures)",
    x_label="structural outcome",
)
fig_s.show()

In [ ]:
layer_s_by_figure = (
    layer_s_df.groupby(["figure", "structural"]).size().unstack(fill_value=0)
)
layer_s_by_figure = layer_s_by_figure.reindex(columns=LAYER_S_ORDER, fill_value=0)
display(layer_s_by_figure)

fig_s_fig = px.bar(
    layer_s_df,
    x="structural",
    color="structural",
    facet_col="figure",
    category_orders={"structural": list(LAYER_S_ORDER)},
    color_discrete_map=OUTCOME_COLORS,
    title="Layer S — panel rows per figure",
    labels={"structural": "outcome", "figure": "figure index"},
)
fig_s_fig.update_layout(showlegend=False)
fig_s_fig.show()

## Layer 1 — applicability reporting

Was the field answered when it should (or should not) have been?

In [ ]:
layer1_summary = counts_to_frame(results["layer1_counts"], LAYER1_ORDER, "layer1")
display(layer1_summary)

layer1_detail = (
    instances_df.groupby("layer1", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("layer1", key=lambda s: s.map({k: i for i, k in enumerate(LAYER1_ORDER)}))
)
display(instances_df.sort_values("instance_path")[["instance_path", "gold", "pred", "layer1"]])

fig_l1 = plot_outcome_bar(
    results["layer1_counts"],
    LAYER1_ORDER,
    title="Layer 1 — applicability outcomes",
    x_label="layer 1 outcome",
)
fig_l1.show()

## Layer 2 — value matching reporting

Only instances with `layer1 = correct_applicable` receive a layer 2 label.
Binary polarity fields use TP/FP/FN/TN; graded strings use match/mismatch.

In [ ]:
layer2_eligible = instances_df[instances_df["layer1"] == "correct_applicable"].copy()
layer2_summary = counts_to_frame(results["layer2_counts"], LAYER2_ORDER, "layer2")
display(layer2_summary)
display(
    layer2_eligible.sort_values("instance_path")[
        ["instance_path", "gold", "pred", "score", "layer2"]
    ]
)

fig_l2 = plot_outcome_bar(
    results["layer2_counts"],
    LAYER2_ORDER,
    title="Layer 2 — matching outcomes (correct_applicable only)",
    x_label="layer 2 outcome",
)
fig_l2.show()

In [ ]:
layer2_by_field = (
    layer2_eligible.assign(
        field=layer2_eligible["instance_path"].str.rsplit(".", n=1).str[-1]
    )
    .groupby(["field", "layer2"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=LAYER2_ORDER, fill_value=0)
)
display(layer2_by_field)

melted = layer2_by_field.reset_index().melt(
    id_vars="field", var_name="layer2", value_name="count"
)
melted = melted[melted["count"] > 0]
fig_l2_field = px.bar(
    melted,
    x="field",
    y="count",
    color="layer2",
    barmode="stack",
    category_orders={"layer2": list(LAYER2_ORDER)},
    color_discrete_map=OUTCOME_COLORS,
    title="Layer 2 outcomes by leaf field",
)
fig_l2_field.show()

## Combined dashboard

Side-by-side summary of all three reporting layers.

In [ ]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("Layer S", "Layer 1", "Layer 2"),
)

for col, (counts, order) in enumerate(
    [
        (results["layer_s_counts"], LAYER_S_ORDER),
        (results["layer1_counts"], LAYER1_ORDER),
        (results["layer2_counts"], LAYER2_ORDER),
    ],
    start=1,
):
    labels = list(order)
    values = [counts.get(label, 0) for label in labels]
    colors = [OUTCOME_COLORS.get(label, "#7f7f7f") for label in labels]
    fig.add_trace(
        go.Bar(x=labels, y=values, marker_color=colors, showlegend=False),
        row=1,
        col=col,
    )

fig.update_layout(
    title_text="Flat evaluation reporting — toy figure checklist",
    height=420,
    yaxis=dict(rangemode="tozero"),
    yaxis2=dict(rangemode="tozero"),
    yaxis3=dict(rangemode="tozero"),
)
fig.show()